In [1]:
# Import

import pandas as pd
import os

In [2]:
# --------------------------------------------------
# 1. Define input files
# --------------------------------------------------

summary_files = {
    "ML": "ml_summary_match_nonmatch.csv",
    "CNN": "cnn_summary_match_nonmatch.csv",
    "BERT_SciBERT": "bert_scibert_summary_match_nonmatch.csv",
    "ChatGPT_ZeroShot": "chatgpt_zero_shot_summary_match_nonmatch.csv",
    "ChatGPT_FewShot": "chatgpt_few_shot_summary_match_nonmatch.csv"
}

In [3]:
# --------------------------------------------------
# 2. Function to read summary files
# --------------------------------------------------

def read_summary_file(file_path):
    # Read a summary CSV with multi-level columns
    df = pd.read_csv(file_path, header=[0, 1], index_col=0)
    
    # Convert multi-index columns into flat column names
    df.columns = [
        f"{metric}_{stat}"
        for metric, stat in df.columns
    ]
    
    # Bring model names from index into a column
    df = df.reset_index()
    df = df.rename(columns={"Model": "Model"})
    
    return df

In [4]:
# --------------------------------------------------
# 3. Load and combine all model summaries
# --------------------------------------------------

summary_dfs = []

for source_name, file_path in summary_files.items():
    temp_df = read_summary_file(file_path)
    temp_df["Source"] = source_name
    summary_dfs.append(temp_df)

combined_summary = pd.concat(summary_dfs, ignore_index=True)

print(combined_summary.head())

                Model  Match_Precision_mean  Match_Precision_std  \
0  LogisticRegression              0.817405             0.040704   
1        RandomForest              0.966585             0.024401   
2                 CNN              0.481224             0.053928   
3                BERT              0.952929             0.039477   
4             SciBERT              0.946492             0.031252   

   Match_Recall_mean  Match_Recall_std  Match_F1_mean  Match_F1_std  \
0           0.913283          0.035409       0.861917      0.027784   
1           0.748653          0.066918       0.842087      0.042104   
2           0.685777          0.056400       0.562290      0.035982   
3           0.929229          0.034625       0.940145      0.024552   
4           0.922149          0.033784       0.933662      0.023296   

   NonMatch_Precision_mean  NonMatch_Precision_std  NonMatch_Recall_mean  \
0                 0.982313                0.006886              0.958657   
1           

In [5]:
# --------------------------------------------------
# 4. Format mean and standard deviation
# --------------------------------------------------

def format_mean_std(mean_value, std_value, digits=4):
    # Format values as mean (std)
    return f"{mean_value:.{digits}f} ({std_value:.{digits}f})"


def build_class_table(combined_df, class_prefix):
    # Build a publication-ready table for Match or NonMatch
    
    rows = []
    
    for _, row in combined_df.iterrows():
        model_name = row["Model"]
        
        precision = format_mean_std(
            row[f"{class_prefix}_Precision_mean"],
            row[f"{class_prefix}_Precision_std"]
        )
        
        recall = format_mean_std(
            row[f"{class_prefix}_Recall_mean"],
            row[f"{class_prefix}_Recall_std"]
        )
        
        f1 = format_mean_std(
            row[f"{class_prefix}_F1_mean"],
            row[f"{class_prefix}_F1_std"]
        )
        
        rows.append({
            "Model": model_name,
            "Precision": precision,
            "Recall": recall,
            "F1": f1
        })
    
    return pd.DataFrame(rows)


# Positive class table
match_table = build_class_table(
    combined_summary,
    class_prefix="Match"
)

# Negative class table
nonmatch_table = build_class_table(
    combined_summary,
    class_prefix="NonMatch"
)

print("Positive (Match) Table")
display(match_table)

print("Negative (NonMatch) Table")
display(nonmatch_table)

Positive (Match) Table


,Model,Precision,Recall,F1
0,LogisticRegression,0.8174 (0.0407),0.9133 (0.0354),0.8619 (0.0278)
1,RandomForest,0.9666 (0.0244),0.7487 (0.0669),0.8421 (0.0421)
2,CNN,0.4812 (0.0539),0.6858 (0.0564),0.5623 (0.0360)
3,BERT,0.9529 (0.0395),0.9292 (0.0346),0.9401 (0.0246)
4,SciBERT,0.9465 (0.0313),0.9221 (0.0338),0.9337 (0.0233)
5,ChatGPT_zero_shot,0.9806 (0.0264),0.7597 (0.0517),0.8550 (0.0319)
6,ChatGPT_few_shot,0.9858 (0.0239),0.7931 (0.0487),0.8779 (0.0278)


Negative (NonMatch) Table


,Model,Precision,Recall,F1
0,LogisticRegression,0.9823 (0.0069),0.9587 (0.0113),0.9703 (0.0063)
1,RandomForest,0.9521 (0.0118),0.9947 (0.0042),0.9729 (0.0059)
2,CNN,0.9312 (0.0103),0.8481 (0.0392),0.8872 (0.0205)
3,BERT,0.9860 (0.0067),0.9905 (0.0088),0.9882 (0.0050)
4,SciBERT,0.9846 (0.0066),0.9894 (0.0065),0.9870 (0.0045)
5,ChatGPT_zero_shot,0.9541 (0.0094),0.9968 (0.0045),0.9750 (0.0049)
6,ChatGPT_few_shot,0.9603 (0.0088),0.9975 (0.0044),0.9785 (0.0043)


In [6]:
# --------------------------------------------------
# 5. Save tables
# --------------------------------------------------

match_table.to_csv(
    "table_positive_match_results.csv",
    index=False,
    encoding="utf-8-sig"
)

nonmatch_table.to_csv(
    "table_negative_nonmatch_results.csv",
    index=False,
    encoding="utf-8-sig"
)